# pandas for Machine Learning — D01–D06

Labeled observations, aligned transformations, explicit missingness, honest groups, validated joins and reproducible audits. This is trusted canonical author code, not hosted learner execution. Notebook execution/download never awards platform mastery. No credentials/uploads/remote data required. Genuine Colab execution remains UNVERIFIED.

Restart and Run All for canonical results. Before each example, predict its labeled rows, types and missingness. Private guided practice may change declared values after predicting the consequences; restart to restore reference outputs.


In [ ]:
import sys, json, hashlib, warnings, time, resource, platform
from importlib.metadata import version
import pandas as pd
import numpy as np
assert sys.version_info[:3] == (3, 13, 16)
profile = {name: version(name) for name in ['numpy', 'pandas', 'matplotlib', 'scikit-learn']}
assert profile == {'numpy': '2.5.4', 'pandas': '3.0.6', 'matplotlib': '3.11.2', 'scikit-learn': '1.9.1'}
started = time.monotonic()
seed = 42
houses = json.loads("{\"id\":[\"H01\",\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\",\"H12\",\"H13\",\"H14\",\"H15\",\"H16\"],\"area\":[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16],\"age\":[0,5,null,10,0,5,null,10,0,5,null,10,0,5,null,10],\"neighborhood\":[\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\",\"A\",\"B\"],\"price\":[100,120,140,160,180,200,220,240,260,280,300,320,340,360,380,400],\"future_price\":[110,130,150,170,190,210,230,250,270,290,310,330,350,370,390,410],\"listed_on\":[\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\",\"2026-01-02\"]}")
manifest = json.loads("{\"id\":\"pandas-houses-teaching.v1\",\"sha256\":\"751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5\",\"source\":\"original fictional data authored for platform teaching\",\"permission\":\"platform teaching use\",\"schema\":{\"id\":\"unique string\",\"area\":\"area units\",\"age\":\"nullable years\",\"neighborhood\":\"label\",\"price\":\"target $k\",\"future_price\":\"quarantined future $k\",\"listed_on\":\"ISO date\"},\"train_ids\":[\"H01\",\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\",\"H12\"],\"test_ids\":[\"H13\",\"H14\",\"H15\",\"H16\"],\"limitations\":\"tiny fictional data; not representative or a capstone\"}")
assert hashlib.sha256(json.dumps(houses, sort_keys=True).encode()).hexdigest() == manifest['sha256']
assert len(set(houses['id'])) == 16
assert manifest['train_ids'] == houses['id'][:12] and manifest['test_ids'] == houses['id'][12:]
all_results = []


In [ ]:
def dtype_name(dtype):
    return f"string[{dtype.storage}]" if isinstance(dtype, pd.StringDtype) else str(dtype)


def normalize(value):
    if isinstance(value, pd.DataFrame):
        assert value.shape[0] <= 32 and value.shape[1] <= 8
        return {"kind": "frame", "shape": list(value.shape), "index": normalize(value.index.tolist()), "index_names": normalize(value.index.names), "columns": normalize(value.columns.tolist()), "dtypes": [dtype_name(dtype) for dtype in value.dtypes], "rows": normalize(value.values.tolist())}
    if isinstance(value, pd.Series):
        assert len(value) <= 32
        return {"kind": "series", "name": normalize(value.name), "index": normalize(value.index.tolist()), "index_names": normalize(value.index.names), "dtype": dtype_name(value.dtype), "values": normalize(value.tolist())}
    if value is pd.NA or value is pd.NaT:
        return {"tag": "missing"}
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, np.generic):
        return normalize(value.item())
    if isinstance(value, float) and not np.isfinite(value):
        return {"tag": "missing"} if np.isnan(value) else {"tag": "nonfinite", "value": "+inf" if value > 0 else "-inf"}
    if isinstance(value, dict):
        return {key: normalize(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [normalize(item) for item in value]
    if isinstance(value, str):
        assert len(value) <= 120
    if isinstance(value, (float, int)) and not isinstance(value, bool):
        assert abs(value) <= 1e12
    return value



## D01 — Series, DataFrames and labeled data

Labels carry identity that positional matrices alone do not.

Recall: P02, P08, N01.

Mathematical connection: Observation/feature matrices plus explicit labels and row identity.

Misconception: A DataFrame is not simply a NumPy matrix with decorative labels.


### D01.O1 — Predict label alignment and preserve row identity.

Predict the output labels, columns, dtypes, missingness and row counts before running. Match index labels, not display positions. A missing label produces missing output, not zero; preserve each observation's identity before interpreting the sum.

Changed-case practice: Draw index-to-value mappings before adding.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    a = pd.Series([1, 2], index=['rowA', 'rowB'], dtype='Int64')
    b = pd.Series([10, 20], index=['rowB', 'rowA'], dtype='Int64')
    result = {'a': a, 'b': b, 'aligned_sum': a + b}
observed = normalize(result)
expected = json.loads("{\"a\":{\"kind\":\"series\",\"name\":null,\"index\":[\"rowA\",\"rowB\"],\"index_names\":[null],\"dtype\":\"Int64\",\"values\":[1,2]},\"b\":{\"kind\":\"series\",\"name\":null,\"index\":[\"rowB\",\"rowA\"],\"index_names\":[null],\"dtype\":\"Int64\",\"values\":[10,20]},\"aligned_sum\":{\"kind\":\"series\",\"name\":null,\"index\":[\"rowA\",\"rowB\"],\"index_names\":[null],\"dtype\":\"Int64\",\"values\":[21,12]}}")
assert observed == expected
all_results.append({'objective': 'D01.O1', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D01.O2 — Read shape, columns, index and dtypes from a small table.

Predict the output labels, columns, dtypes, missingness and row counts before running. Shape counts rows and columns; the index names observations and columns name ordered features. Inspect all four metadata fields before selecting model input.

Changed-case practice: Use the metadata panel, not visual row numbers.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    df = pd.DataFrame({'area': [1, 2, 3], 'age': [0, 10, 20]}, index=[10, 20, 30])
    result = {'shape': df.shape, 'columns': df.columns.tolist(), 'index': df.index.tolist(), 'dtypes': [str(dtype) for dtype in df.dtypes]}
observed = normalize(result)
expected = json.loads("{\"shape\":[3,2],\"columns\":[\"area\",\"age\"],\"index\":[10,20,30],\"dtypes\":[\"int64\",\"int64\"]}")
assert observed == expected
all_results.append({'objective': 'D01.O2', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D01.O3 — Distinguish a Series selection from a one-column DataFrame.

Predict the output labels, columns, dtypes, missingness and row counts before running. A single column name selects a one-dimensional labeled Series. A list of column names preserves a two-dimensional DataFrame, even with one column.

Changed-case practice: Compare df['area'] with df[['area']].


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    df = pd.DataFrame({'area': [1, 2, 3], 'age': [0, 10, 20]}, index=[10, 20, 30])
    result = {'series': df['area'], 'frame': df[['area']]}
observed = normalize(result)
expected = json.loads("{\"series\":{\"kind\":\"series\",\"name\":\"area\",\"index\":[10,20,30],\"index_names\":[null],\"dtype\":\"int64\",\"values\":[1,2,3]},\"frame\":{\"kind\":\"frame\",\"shape\":[3,1],\"index\":[10,20,30],\"index_names\":[null],\"columns\":[\"area\"],\"dtypes\":[\"int64\"],\"rows\":[[1],[2],[3]]}}")
assert observed == expected
all_results.append({'objective': 'D01.O3', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D01.O4 — Identify nullable numeric/string data without inventing zeros.

Predict the output labels, columns, dtypes, missingness and row counts before running. Nullable Int64 and explicit string storage preserve missing values. Observed zero age is not missing, and an unknown label is not the string 'None'.

Changed-case practice: Review explicit schema/nullability definitions.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    df = pd.DataFrame({'age': pd.Series([0, None, 21], dtype='Int64'), 'zone': pd.Series(['A', None, 'B'], dtype='string[python]')})
    result = {'table': df, 'missing': df.isna()}
observed = normalize(result)
expected = json.loads("{\"table\":{\"kind\":\"frame\",\"shape\":[3,2],\"index\":[0,1,2],\"index_names\":[null],\"columns\":[\"age\",\"zone\"],\"dtypes\":[\"Int64\",\"string[python]\"],\"rows\":[[0,\"A\"],[{\"tag\":\"missing\"},{\"tag\":\"missing\"}],[21,\"B\"]]},\"missing\":{\"kind\":\"frame\",\"shape\":[3,2],\"index\":[0,1,2],\"index_names\":[null],\"columns\":[\"age\",\"zone\"],\"dtypes\":[\"bool\",\"bool\"],\"rows\":[[false,false],[true,true],[false,false]]}}")
assert observed == expected
all_results.append({'objective': 'D01.O4', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


## D02 — Selecting and filtering without losing alignment

Selection is a cohort and feature-contract decision, not just bracket syntax.

Recall: D01, N02, P03.

Mathematical connection: A filter is a selection function; masks have identity as well as length.

Misconception: .loc[0] need not mean the first row; a filter can be valid Python but wrong for the intended cohort.


### D02.O1 — Predict .loc and .iloc values with nontrivial indexes.

Predict the output labels, columns, dtypes, missingness and row counts before running. loc addresses an index label; iloc addresses a zero-based position. They coincide only when that position happens to carry the requested label.

Changed-case practice: Label positions separately from index values.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    df = pd.DataFrame({'area': [1, 2, 3], 'age': [0, 10, 20]}, index=[10, 20, 30])
    result = {'label20': df.loc[20], 'position': df.iloc[1]}
observed = normalize(result)
expected = json.loads("{\"label20\":{\"kind\":\"series\",\"name\":20,\"index\":[\"area\",\"age\"],\"index_names\":[null],\"dtype\":\"int64\",\"values\":[2,10]},\"position\":{\"kind\":\"series\",\"name\":20,\"index\":[\"area\",\"age\"],\"index_names\":[null],\"dtype\":\"int64\",\"values\":[2,10]}}")
assert observed == expected
all_results.append({'objective': 'D02.O1', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D02.O2 — Apply the intended filter to aligned observations/targets.

Predict the output labels, columns, dtypes, missingness and row counts before running. The Boolean Series aligns by label: retain IDs20 and30, with their own targets200 and300. Taking the first two targets would produce a silently different training table.

Changed-case practice: Compare the mask index with the table index.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    df = pd.DataFrame({'area': [1, 2, 3], 'age': [0, 10, 20]}, index=[10, 20, 30])
    y = pd.Series([100, 200, 300], index=[10, 20, 30], dtype='Int64')
    mask = pd.Series([True, False, True], index=[30, 10, 20], dtype='boolean')
    result = {'source': df, 'mask': mask, 'selected': df.loc[mask], 'targets': y.loc[mask]}
observed = normalize(result)
expected = json.loads("{\"source\":{\"kind\":\"frame\",\"shape\":[3,2],\"index\":[10,20,30],\"index_names\":[null],\"columns\":[\"area\",\"age\"],\"dtypes\":[\"int64\",\"int64\"],\"rows\":[[1,0],[2,10],[3,20]]},\"mask\":{\"kind\":\"series\",\"name\":null,\"index\":[30,10,20],\"index_names\":[null],\"dtype\":\"boolean\",\"values\":[true,false,true]},\"selected\":{\"kind\":\"frame\",\"shape\":[2,2],\"index\":[20,30],\"index_names\":[null],\"columns\":[\"area\",\"age\"],\"dtypes\":[\"int64\",\"int64\"],\"rows\":[[2,10],[3,20]]},\"targets\":{\"kind\":\"series\",\"name\":null,\"index\":[20,30],\"index_names\":[null],\"dtype\":\"Int64\",\"values\":[200,300]}}")
assert observed == expected
all_results.append({'objective': 'D02.O2', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D02.O3 — Select columns without silently changing feature order.

Predict the output labels, columns, dtypes, missingness and row counts before running. A feature list establishes coefficient order. Selecting the same names in a different order changes the model's input contract, even if table shape stays identical.

Changed-case practice: Match the downstream coefficient names.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    df = pd.DataFrame({'area': [1, 2, 3], 'age': [0, 10, 20]}, index=[10, 20, 30])
    columns = ['area', 'age']
    result = df.loc[:, columns]
observed = normalize(result)
expected = json.loads("{\"kind\":\"frame\",\"shape\":[3,2],\"index\":[10,20,30],\"index_names\":[null],\"columns\":[\"area\",\"age\"],\"dtypes\":[\"int64\",\"int64\"],\"rows\":[[1,0],[2,10],[3,20]]}")
assert observed == expected
all_results.append({'objective': 'D02.O3', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D02.O4 — Use explicit .loc assignment or .copy() under the pinned pandas behavior.

Predict the output labels, columns, dtypes, missingness and row counts before running. Explicitly copy the frame that you intend to edit, then assign with loc. The original remains unchanged; do not rely on chained assignment side effects under this pinned pandas version.

Changed-case practice: Trace the deliberately owned frame and resulting values.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    df = pd.DataFrame({'area': [1, 2, 3], 'age': [0, 10, 20]}, index=[10, 20, 30])
    owned = df.copy()
    owned.loc[20, 'area'] = 9
    result = {'original': df, 'owned_copy': owned}
observed = normalize(result)
expected = json.loads("{\"original\":{\"kind\":\"frame\",\"shape\":[3,2],\"index\":[10,20,30],\"index_names\":[null],\"columns\":[\"area\",\"age\"],\"dtypes\":[\"int64\",\"int64\"],\"rows\":[[1,0],[2,10],[3,20]]},\"owned_copy\":{\"kind\":\"frame\",\"shape\":[3,2],\"index\":[10,20,30],\"index_names\":[null],\"columns\":[\"area\",\"age\"],\"dtypes\":[\"int64\",\"int64\"],\"rows\":[[1,0],[9,10],[3,20]]}}")
assert observed == expected
all_results.append({'objective': 'D02.O4', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


## D03 — Missing values require a stated policy

A missingness policy changes data and assumptions, not only cell appearance.

Recall: D01–D02, N06.

Mathematical connection: Denominators and summaries change when observations are removed.

Misconception: Empty-looking values, zeros and missing labels are not equivalent; imputation is an assumption.


### D03.O1 — Detect missingness without discarding valid zero values.

Predict the output labels, columns, dtypes, missingness and row counts before running. isna identifies only the absent observation. notna retains the legitimate zero H1 as well as H3; truthiness is not a missing-value policy.

Changed-case practice: Use isna masks beside original values.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    ages = pd.Series([0, None, 21], index=['H1', 'H2', 'H3'], dtype='Int64')
    result = {'ages': ages, 'missing': ages.isna(), 'observed': ages.loc[ages.notna()]}
observed = normalize(result)
expected = json.loads("{\"ages\":{\"kind\":\"series\",\"name\":null,\"index\":[\"H1\",\"H2\",\"H3\"],\"index_names\":[null],\"dtype\":\"Int64\",\"values\":[0,{\"tag\":\"missing\"},21]},\"missing\":{\"kind\":\"series\",\"name\":null,\"index\":[\"H1\",\"H2\",\"H3\"],\"index_names\":[null],\"dtype\":\"bool\",\"values\":[false,true,false]},\"observed\":{\"kind\":\"series\",\"name\":null,\"index\":[\"H1\",\"H3\"],\"index_names\":[null],\"dtype\":\"Int64\",\"values\":[0,21]}}")
assert observed == expected
all_results.append({'objective': 'D03.O1', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D03.O2 — Compute missingness counts/rates with the correct denominator.

Predict the output labels, columns, dtypes, missingness and row counts before running. The missingness rate uses all 4 eligible rows: 2/4. Do not divide by only nonmissing records or count the legitimate zero as missing.

Changed-case practice: Count all eligible rows before dividing.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    ages = pd.Series([0, None, 1, None], dtype='Int64')
    result = {'rows': len(ages), 'missing_count': int(ages.isna().sum()), 'missing_rate': float(ages.isna().mean())}
observed = normalize(result)
expected = json.loads("{\"rows\":4,\"missing_count\":2,\"missing_rate\":0.5}")
assert observed == expected
all_results.append({'objective': 'D03.O2', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D03.O3 — Predict fillna/dropna effects on values and shape.

Predict the output labels, columns, dtypes, missingness and row counts before running. Dropping removes H2 and changes the population; filling retains its ID and records an assumption. In the worked case10 is the observed median of0 and20, not a permission to fit on a future test set.

Changed-case practice: Compare retained row IDs and replacement values.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    ages = pd.Series([0, None, 20], index=['H1', 'H2', 'H3'], dtype='Int64')
    replacement = 10
    result = {'preserved': ages, 'dropped': ages.dropna(), 'declared_fill': ages.fillna(replacement)}
observed = normalize(result)
expected = json.loads("{\"preserved\":{\"kind\":\"series\",\"name\":null,\"index\":[\"H1\",\"H2\",\"H3\"],\"index_names\":[null],\"dtype\":\"Int64\",\"values\":[0,{\"tag\":\"missing\"},20]},\"dropped\":{\"kind\":\"series\",\"name\":null,\"index\":[\"H1\",\"H3\"],\"index_names\":[null],\"dtype\":\"Int64\",\"values\":[0,20]},\"declared_fill\":{\"kind\":\"series\",\"name\":null,\"index\":[\"H1\",\"H2\",\"H3\"],\"index_names\":[null],\"dtype\":\"Int64\",\"values\":[0,10,20]}}")
assert observed == expected
all_results.append({'objective': 'D03.O3', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D03.O4 — Separate exploratory filling from training-fitted preprocessing.

Predict the output labels, columns, dtypes, missingness and row counts before running. The replacement is a declared statistic from training rowsH1/H2 only. Transform held-out rows with that saved value; computing it from the whole dataset before a split lets held-out information influence the model.

Changed-case practice: Revisit the earlier Statistics leakage lesson and the local fit-boundary explanation; T02 is an optional later extension, not a prerequisite.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    train_ids = ['H1', 'H2']
    test_ids = ['H3', 'H4']
    training_replacement = 10
    test = pd.Series([1001, None], index=test_ids, dtype='Int64')
    result = {'fit_rows': train_ids, 'transformed_test': test.fillna(training_replacement), 'refit_on_test': False}
observed = normalize(result)
expected = json.loads("{\"fit_rows\":[\"H1\",\"H2\"],\"transformed_test\":{\"kind\":\"series\",\"name\":null,\"index\":[\"H3\",\"H4\"],\"index_names\":[null],\"dtype\":\"Int64\",\"values\":[1001,10]},\"refit_on_test\":false}")
assert observed == expected
all_results.append({'objective': 'D03.O4', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


## D04 — Grouping, aggregating and reporting denominators

Grouped summaries have group-specific denominators and cannot be averaged blindly.

Recall: D02–D03; Statistics means.

Mathematical connection: Group means, weighted totals and conditional summaries.

Misconception: A mean of group means need not equal the global mean; count excludes missing values while size counts rows.


### D04.O1 — Predict an explicitly configured groupby aggregation.

Predict the output labels, columns, dtypes, missingness and row counts before running. List group membership first, then compute the selected aggregation. Explicit sorted groups and denominators make the report reproducible, not a statement about the whole population.

Changed-case practice: List membership before calculating.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    df = pd.DataFrame({'group': ['A', 'A', 'B'], 'price': [10, 20, 100]})
    result = df.groupby('group', sort=True, dropna=False, observed=True)['price'].agg(['sum', 'mean', 'count', 'size'])
observed = normalize(result)
expected = json.loads("{\"kind\":\"frame\",\"shape\":[2,4],\"index\":[\"A\",\"B\"],\"index_names\":[\"group\"],\"columns\":[\"sum\",\"mean\",\"count\",\"size\"],\"dtypes\":[\"int64\",\"float64\",\"int64\",\"int64\"],\"rows\":[[30,15,2,2],[100,100,1,1]]}")
assert observed == expected
all_results.append({'objective': 'D04.O1', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D04.O2 — Use correct group denominators and distinguish size from count.

Predict the output labels, columns, dtypes, missingness and row counts before running. size counts all rows, but count excludes missing price. GroupA has3 rows and2 observed prices; reconstruct the total sum and observed count for the global mean instead of averaging unequal group means.

Changed-case practice: Reconstruct sum and count for each group.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    df = pd.DataFrame({'group': ['A', 'A', 'A', 'B'], 'price': [10, 20, None, 100]})
    summary = df.groupby('group', sort=True, dropna=False, observed=True)['price'].agg(['sum', 'mean', 'count', 'size'])
    result = {'groups': summary, 'global_mean': df['price'].mean(), 'unweighted_group_mean': summary['mean'].mean()}
observed = normalize(result)
expected = json.loads("{\"groups\":{\"kind\":\"frame\",\"shape\":[2,4],\"index\":[\"A\",\"B\"],\"index_names\":[\"group\"],\"columns\":[\"sum\",\"mean\",\"count\",\"size\"],\"dtypes\":[\"float64\",\"float64\",\"int64\",\"int64\"],\"rows\":[[30,15,2,3],[100,100,1,1]]},\"global_mean\":43.333333333333336,\"unweighted_group_mean\":57.5}")
assert observed == expected
all_results.append({'objective': 'D04.O2', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D04.O3 — Handle missing keys and category inclusion explicitly.

Predict the output labels, columns, dtypes, missingness and row counts before running. The recipe explicitly decides whether missing keys are dropped and unused categories are included. observed and dropna describe different choices; inspect the category universe and actual groups separately.

Changed-case practice: Compare groups included versus excluded by the policy.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    df = pd.DataFrame({'group': pd.Categorical(['A', None, 'A'], categories=['A', 'B']), 'price': [1, 20, 30]})
    result = df.groupby('group', sort=True, dropna=False, observed=True)['price'].agg(['sum', 'count', 'size'])
observed = normalize(result)
expected = json.loads("{\"kind\":\"frame\",\"shape\":[2,3],\"index\":[\"A\",{\"tag\":\"missing\"}],\"index_names\":[\"group\"],\"columns\":[\"sum\",\"count\",\"size\"],\"dtypes\":[\"int64\",\"int64\",\"int64\"],\"rows\":[[31,2,2],[20,1,1]]}")
assert observed == expected
all_results.append({'objective': 'D04.O3', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D04.O4 — Avoid causal conclusions from group differences.

Predict the output labels, columns, dtypes, missingness and row counts before running. The pooled means differ because the size-band composition differs. Within each band the group prices agree in this constructed data. Grouping describes association; neither result alone identifies a causal effect or proves no causal effect in another study.

Changed-case practice: Return to the Statistics observational/intervention experiment.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    df = pd.DataFrame({'group': ['A', 'A', 'A', 'B', 'B', 'B'], 'band': ['small', 'small', 'large', 'small', 'large', 'large'], 'price': [101, 101, 301, 101, 301, 301]})
    result = {'pooled': df.groupby('group', sort=True, dropna=False, observed=True)['price'].mean(), 'within_band': df.groupby(['band', 'group'], sort=True, dropna=False, observed=True)['price'].mean()}
observed = normalize(result)
expected = json.loads("{\"pooled\":{\"kind\":\"series\",\"name\":\"price\",\"index\":[\"A\",\"B\"],\"index_names\":[\"group\"],\"dtype\":\"float64\",\"values\":[167.66666666666666,234.33333333333334]},\"within_band\":{\"kind\":\"series\",\"name\":\"price\",\"index\":[[\"large\",\"A\"],[\"large\",\"B\"],[\"small\",\"A\"],[\"small\",\"B\"]],\"index_names\":[\"band\",\"group\"],\"dtype\":\"float64\",\"values\":[301,301,101,101]}}")
assert observed == expected
all_results.append({'objective': 'D04.O4', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


## D05 — Joins can duplicate or lose observations

Joining constructs matching pairs and can change effective observation weights.

Recall: D01–D04.

Mathematical connection: Relational matching changes sample weights and row multiplicities.

Misconception: Joining does not guarantee one row per customer; equal column names do not prove a valid key.


### D05.O1 — Choose valid join keys and an explicit matching contract.

Predict the output labels, columns, dtypes, missingness and row counts before running. Entity IDs identify the same house across tables. Joining on zone creates cross-house pairs. Neither a permitted cardinality nor an accidentally correct sample establishes key meaning.

Changed-case practice: Inspect uniqueness, meaning and missing keys.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    left = pd.DataFrame({'entity_id': ['H31', 'H32'], 'zone': ['X', 'X']}, dtype='string[python]')
    right = pd.DataFrame({'entity_id': ['H31', 'H32'], 'zone': ['X', 'X'], 'plan': ['basic', 'plus']}, dtype='string[python]')
    result = {'entity_matches': left.merge(right, on='entity_id', how='left', sort=False, validate='one_to_one'), 'zone_matches': left.merge(right, on='zone', how='left', sort=False, validate='many_to_many')}
observed = normalize(result)
expected = json.loads("{\"entity_matches\":{\"kind\":\"frame\",\"shape\":[2,4],\"index\":[0,1],\"index_names\":[null],\"columns\":[\"entity_id\",\"zone_x\",\"zone_y\",\"plan\"],\"dtypes\":[\"string[python]\",\"string[python]\",\"string[python]\",\"string[python]\"],\"rows\":[[\"H31\",\"X\",\"X\",\"basic\"],[\"H32\",\"X\",\"X\",\"plus\"]]},\"zone_matches\":{\"kind\":\"frame\",\"shape\":[4,4],\"index\":[0,1,2,3],\"index_names\":[null],\"columns\":[\"entity_id_x\",\"zone\",\"entity_id_y\",\"plan\"],\"dtypes\":[\"string[python]\",\"string[python]\",\"string[python]\",\"string[python]\"],\"rows\":[[\"H31\",\"X\",\"H31\",\"basic\"],[\"H31\",\"X\",\"H32\",\"plus\"],[\"H32\",\"X\",\"H31\",\"basic\"],[\"H32\",\"X\",\"H32\",\"plus\"]]}}")
assert observed == expected
all_results.append({'objective': 'D05.O1', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D05.O2 — Predict duplication/loss and reject unexpected many-to-many joins.

Predict the output labels, columns, dtypes, missingness and row counts before running. The intended lookup has at most one right-side record per key. Each A pairs with both conflicting right records; many_to_one correctly rejects this lookup. A permissive many_to_many preview counts pairs but cannot justify silently duplicating observations or dropping conflicting records.

Changed-case practice: Draw the matching pairs, then count multiplicities.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    left = pd.DataFrame({'key': ['A', 'B']}, dtype='string[python]')
    right = pd.DataFrame({'key': ['A', 'A'], 'lookup': ['x', 'y']}, dtype='string[python]')
    preview = left.merge(right, on='key', how='left', sort=False, validate='many_to_many', indicator=True)
    try:
        checked = left.merge(right, on='key', how='left', sort=False, validate='many_to_one')
    except pd.errors.MergeError:
        validation = {'tag': 'error', 'class': 'MergeError'}
    result = {'preview': preview, 'required_lookup_contract': validation, 'action': 'inspect conflicting right keys before use'}
observed = normalize(result)
expected = json.loads("{\"preview\":{\"kind\":\"frame\",\"shape\":[3,3],\"index\":[0,1,2],\"index_names\":[null],\"columns\":[\"key\",\"lookup\",\"_merge\"],\"dtypes\":[\"string[python]\",\"string[python]\",\"category\"],\"rows\":[[\"A\",\"x\",\"both\"],[\"A\",\"y\",\"both\"],[\"B\",{\"tag\":\"missing\"},\"left_only\"]]},\"required_lookup_contract\":{\"tag\":\"error\",\"class\":\"MergeError\"},\"action\":\"inspect conflicting right keys before use\"}")
assert observed == expected
all_results.append({'objective': 'D05.O2', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D05.O3 — Distinguish inner from left join and inspect unmatched rows.

Predict the output labels, columns, dtypes, missingness and row counts before running. A left join retains the requested left population, including unmatched rows; an inner join drops unmatched rows. Each duplicate key creates another matching pair. Inspect exact IDs, indicator values and multiplicities before interpreting the result.

Changed-case practice: Compare the requested population before/after joining.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    left = pd.DataFrame({'key': ['A', 'B']}, dtype='string[python]')
    right = pd.DataFrame({'key': ['A', 'A'], 'value': ['x', 'y']}, dtype='string[python]')
    result = {'left': left.merge(right, on='key', how='left', sort=False, validate='one_to_many', indicator=True), 'inner': left.merge(right, on='key', how='inner', sort=False, validate='one_to_many', indicator=True)}
observed = normalize(result)
expected = json.loads("{\"left\":{\"kind\":\"frame\",\"shape\":[3,3],\"index\":[0,1,2],\"index_names\":[null],\"columns\":[\"key\",\"value\",\"_merge\"],\"dtypes\":[\"string[python]\",\"string[python]\",\"category\"],\"rows\":[[\"A\",\"x\",\"both\"],[\"A\",\"y\",\"both\"],[\"B\",{\"tag\":\"missing\"},\"left_only\"]]},\"inner\":{\"kind\":\"frame\",\"shape\":[2,3],\"index\":[0,1],\"index_names\":[null],\"columns\":[\"key\",\"value\",\"_merge\"],\"dtypes\":[\"string[python]\",\"string[python]\",\"category\"],\"rows\":[[\"A\",\"x\",\"both\"],[\"A\",\"y\",\"both\"]]}}")
assert observed == expected
all_results.append({'objective': 'D05.O3', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D05.O4 — Recognize that pandas null-key matching needs explicit handling.

Predict the output labels, columns, dtypes, missingness and row counts before running. When both sides contain null keys, pinned pandas matches them. Without a right null there is no such pair. Do not assume SQL-style null nonmatches; explicitly inspect whether missing keys can identify the same entity.

Changed-case practice: Inspect the pinned merge null-key fixture.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    left = pd.DataFrame({'key': ['A', None], 'left_id': ['L31', 'L32']}, dtype='string[python]')
    right = pd.DataFrame({'key': [None, 'A'], 'right_id': ['R31', 'R32']}, dtype='string[python]')
    result = left.merge(right, on='key', how='inner', sort=False, validate='one_to_one', indicator=True)
observed = normalize(result)
expected = json.loads("{\"kind\":\"frame\",\"shape\":[2,4],\"index\":[0,1],\"index_names\":[null],\"columns\":[\"key\",\"left_id\",\"right_id\",\"_merge\"],\"dtypes\":[\"string[python]\",\"string[python]\",\"string[python]\",\"category\"],\"rows\":[[\"A\",\"L31\",\"R32\",\"both\"],[{\"tag\":\"missing\"},\"L32\",\"R31\",\"both\"]]}")
assert observed == expected
all_results.append({'objective': 'D05.O4', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


## D06 — A reproducible dataset audit

An audit asks whether rows/features are usable for a stated prediction-time question.

Recall: D01–D05, P08; Statistics leakage recall.

Mathematical connection: Summary statistics are meaningful only for the stated rows, units and split.

Misconception: A dataframe that loads successfully is not automatically suitable model input.


### D06.O1 — Inspect bundled CSV types, dates and missing markers deterministically.

Predict the output labels, columns, dtypes, missingness and row counts before running. The loader declares dtype and blank markers; the date parser declares format and coercion. A coerced invalid date is explicitly missing and must be audited, not treated as a successfully validated date or an area zero.

Changed-case practice: Compare raw strings with the declared schema.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    from io import StringIO
    text = 'id,area,listed_on\nH1,1,2026-01-02\nH2,,bad-date\n'
    df = pd.read_csv(StringIO(text), dtype={'id': 'string[python]', 'area': 'Float64', 'listed_on': 'string[python]'}, keep_default_na=False, na_values=[''])
    df['listed_on'] = pd.to_datetime(df['listed_on'], format='%Y-%m-%d', errors='coerce')
    result = {'table': df, 'invalid_dates': int(df['listed_on'].isna().sum()), 'missing_area': int(df['area'].isna().sum())}
observed = normalize(result)
expected = json.loads("{\"table\":{\"kind\":\"frame\",\"shape\":[2,3],\"index\":[0,1],\"index_names\":[null],\"columns\":[\"id\",\"area\",\"listed_on\"],\"dtypes\":[\"string[python]\",\"Float64\",\"datetime64[us]\"],\"rows\":[[\"H1\",1,\"2026-01-02T00:00:00\"],[\"H2\",{\"tag\":\"missing\"},{\"tag\":\"missing\"}]]},\"invalid_dates\":1,\"missing_area\":1}")
assert observed == expected
all_results.append({'objective': 'D06.O1', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D06.O2 — Report duplicates/missingness/group counts using row identities.

Predict the output labels, columns, dtypes, missingness and row counts before running. The named overlay has16 rows but15 distinct IDs; inspect both duplicate occurrences and missing-age counts. The canonical16 unique house IDs remain unchanged. Equal row count does not prove unique observations.

Changed-case practice: Revisit D03–D05 counts and key checks.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    df = pd.DataFrame({'id': ['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16'], 'area': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16], 'age': [0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10], 'neighborhood': ['A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B'], 'price': [100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400], 'future_price': [110, 130, 150, 170, 190, 210, 230, 250, 270, 290, 310, 330, 350, 370, 390, 410], 'listed_on': ['2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02']})
    df['id'] = pd.Series(df['id'], dtype='string[python]')
    df['age'] = pd.Series(df['age'], dtype='Int64')
    df['neighborhood'] = pd.Series(df['neighborhood'], dtype='string[python]')
    overlay = df.copy()
    overlay.loc[10, 'id'] = 'H01'
    result = {'overlay': 'duplicate-id-3', 'rows': len(overlay), 'distinct_ids': len(overlay['id'].drop_duplicates()), 'duplicate_rows': overlay.loc[overlay['id'].duplicated(keep=False), ['id', 'area']], 'missing_age': int(overlay['age'].isna().sum()), 'group_rows': overlay.groupby('neighborhood', sort=True, dropna=False, observed=True).size()}
observed = normalize(result)
expected = json.loads("{\"overlay\":\"duplicate-id-3\",\"rows\":16,\"distinct_ids\":15,\"duplicate_rows\":{\"kind\":\"frame\",\"shape\":[2,2],\"index\":[0,10],\"index_names\":[null],\"columns\":[\"id\",\"area\"],\"dtypes\":[\"string[python]\",\"int64\"],\"rows\":[[\"H01\",1],[\"H01\",11]]},\"missing_age\":4,\"group_rows\":{\"kind\":\"series\",\"name\":null,\"index\":[\"A\",\"B\"],\"index_names\":[\"neighborhood\"],\"dtype\":\"int64\",\"values\":[8,8]}}")
assert observed == expected
all_results.append({'objective': 'D06.O2', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D06.O3 — Exclude targets and unavailable-at-prediction columns from features.

Predict the output labels, columns, dtypes, missingness and row counts before running. Use only fields available at prediction time, in the declared feature order. price is the target; future_price is unavailable then. Exclude both even if highly predictive, and still resolve missingness before fitting a model.

Changed-case practice: Build the feature-availability timeline.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    df = pd.DataFrame({'id': ['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16'], 'area': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16], 'age': [0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10], 'neighborhood': ['A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B'], 'price': [100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400], 'future_price': [110, 130, 150, 170, 190, 210, 230, 250, 270, 290, 310, 330, 350, 370, 390, 410], 'listed_on': ['2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02']})
    df['id'] = pd.Series(df['id'], dtype='string[python]')
    df['age'] = pd.Series(df['age'], dtype='Int64')
    df['neighborhood'] = pd.Series(df['neighborhood'], dtype='string[python]')
    features = ['area', 'age']
    result = {'ordered_features': features, 'inputs': df.loc[:2, features], 'excluded': ['price', 'future_price'], 'ready_to_fit': False}
observed = normalize(result)
expected = json.loads("{\"ordered_features\":[\"area\",\"age\"],\"inputs\":{\"kind\":\"frame\",\"shape\":[3,2],\"index\":[0,1,2],\"index_names\":[null],\"columns\":[\"area\",\"age\"],\"dtypes\":[\"int64\",\"Int64\"],\"rows\":[[1,0],[2,5],[3,{\"tag\":\"missing\"}]]},\"excluded\":[\"price\",\"future_price\"],\"ready_to_fit\":false}")
assert observed == expected
all_results.append({'objective': 'D06.O3', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


### D06.O4 — Produce a bounded audit record with provenance and limitations.

Predict the output labels, columns, dtypes, missingness and row counts before running. A reproducible audit distinguishes its16-row source from its reported subset, records source hash, exact retained/split IDs, missingness and limitations. This fictional sample cannot establish population performance; an audit is not a fitted model or capstone.

Changed-case practice: Use the dataset manifest and exact retained IDs.


In [ ]:
with warnings.catch_warnings(record=True) as observed_warnings:
    warnings.simplefilter('always')
    import pandas as pd
    df = pd.DataFrame({'id': ['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16'], 'area': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16], 'age': [0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10, 0, 5, None, 10], 'neighborhood': ['A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B', 'A', 'B'], 'price': [100, 120, 140, 160, 180, 200, 220, 240, 260, 280, 300, 320, 340, 360, 380, 400], 'future_price': [110, 130, 150, 170, 190, 210, 230, 250, 270, 290, 310, 330, 350, 370, 390, 410], 'listed_on': ['2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02', '2026-01-02']})
    df['id'] = pd.Series(df['id'], dtype='string[python]')
    df['age'] = pd.Series(df['age'], dtype='Int64')
    df['neighborhood'] = pd.Series(df['neighborhood'], dtype='string[python]')
    result = {'dataset': 'pandas-houses-teaching.v1', 'source_sha256': '751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5', 'source_rows': 16, 'reported_rows': len(df), 'row_ids': df['id'].tolist(), 'columns': df.columns.tolist(), 'missing_age': int(df['age'].isna().sum()), 'train_ids': df.loc[:11, 'id'].tolist(), 'test_ids': df.loc[12:, 'id'].tolist(), 'limitations': 'fictional teaching data; report covers only declared retained IDs, not population performance'}
observed = normalize(result)
expected = json.loads("{\"dataset\":\"pandas-houses-teaching.v1\",\"source_sha256\":\"751b55a15cf32f3755684a24e167fce0aa732d3fac7c5246bc86723941131cd5\",\"source_rows\":16,\"reported_rows\":16,\"row_ids\":[\"H01\",\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\",\"H12\",\"H13\",\"H14\",\"H15\",\"H16\"],\"columns\":[\"id\",\"area\",\"age\",\"neighborhood\",\"price\",\"future_price\",\"listed_on\"],\"missing_age\":4,\"train_ids\":[\"H01\",\"H02\",\"H03\",\"H04\",\"H05\",\"H06\",\"H07\",\"H08\",\"H09\",\"H10\",\"H11\",\"H12\"],\"test_ids\":[\"H13\",\"H14\",\"H15\",\"H16\"],\"limitations\":\"fictional teaching data; report covers only declared retained IDs, not population performance\"}")
assert observed == expected
all_results.append({'objective': 'D06.O4', 'outcome': observed, 'warnings': sorted({type(item.message).__name__ for item in observed_warnings})})
print(json.dumps(all_results[-1], indent=2, allow_nan=False))


## Independent analytic anchors and audit limitations

A label-preserving calculation, honest denominator and valid join syntax can still answer the wrong ML question. Inspect source IDs, feature availability, cardinality and missingness policy before concluding that data are usable. This fictional notebook does not validate population performance or causal effects.


In [ ]:
a = pd.Series([1, 2], index=['rowA', 'rowB'], dtype='Int64')
b = pd.Series([10, 20], index=['rowB', 'rowA'], dtype='Int64')
assert (a + b).tolist() == [21, 12]
groups = pd.DataFrame({'group': ['A', 'A', 'B'], 'value': [10, 20, 100]})
means = groups.groupby('group', sort=True, dropna=False, observed=True)['value'].mean()
assert means.tolist() == [15, 100]
assert abs(groups['value'].mean() - 130 / 3) <= 1e-9
assert means.mean() == 57.5
left = pd.DataFrame({'key': ['A', 'B']})
right = pd.DataFrame({'key': ['A', 'A']})
assert len(left.merge(right, on='key', how='left', sort=False, validate='one_to_many')) == 3
assert len(left.merge(right, on='key', how='inner', sort=False, validate='one_to_many')) == 2
assert pd.Series([0, None, 20], dtype='Int64').fillna(10).tolist() == [0, 10, 20]
assert len(all_results) == 24 and len({item['objective'] for item in all_results}) == 24
assert time.monotonic() - started < 300
peak_rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
assert peak_rss < 2 * 1024**3
summary = {'families': 24, 'lessons': 6, 'seed': seed, 'profile': profile, 'manifest': manifest, 'anchors': {'aligned_sum': [21, 12], 'group_means': [15, 100], 'global_mean': 130 / 3, 'left_inner_rows': [3, 2], 'filled_ages': [0, 10, 20]}, 'examples': all_results}
assert len(json.dumps(summary, allow_nan=False).encode()) <= 1024**2
print(json.dumps(summary, sort_keys=True, allow_nan=False))
